# ZenteiQ AI Tech Innovations — Task 2: Dense Model (Qwen)
## MaxText Qwen3-0.6B CPU Training Run & Benchmark

This notebook provides a detailed setup and execution workflow for training the **Qwen3-0.6B** base model on a **CPU-only** environment using **MaxText** and **JAX** with synthetic data.

### 1. Installation & Environment Setup

We will clone the MaxText repository and install its dependencies. When running on a CPU-only environment (such as Google Colab with the CPU runtime selected), JAX defaults to CPU execution. We will install the dependencies using `uv` for fast installations.

In [8]:
!ls

drive  sample_data


In [ ]:
# Clone the MaxText repository
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

# Install uv (fast package installer)
!pip install uv

# Set environmental variables to avoid GPU/TPU requirements
import os
os.environ["UV_TORCH_BACKEND"] = "cpu"

# Install MaxText CPU dependencies
!uv pip install -e .

# Restart the runtime after this step if run in Colab!

: 

In [10]:
!uv pip install qwix -q

In [11]:
!uv pip install tokamax -q

### 2. Verify JAX CPU Backend Connection

Run the following cell to confirm that JAX successfully detects the CPU backend.

In [12]:
!ls

AUTHORS      CONTRIBUTING.md  LICENSE_HEADER  pyproject.toml  scripts  tools
benchmarks   docs	      PREFLIGHT.md    pytest.ini      src
codecov.yml  LICENSE	      pylintrc	      README.md       tests


In [13]:
import jax
print("JAX Version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default backend:", jax.default_backend())

JAX Version: 0.10.1
Available devices: [CpuDevice(id=0)]
Default backend: cpu


### 3. Configuration Breakdown

Below are the parameters we are setting for this training run, along with their purpose and effect:

| Configuration Parameter | Value | Purpose | Effect |
| :--- | :--- | :--- | :--- |
| **`model_name`** | `qwen3-0.6b` | Specifies the base architecture layout to use. | Loads structural config from `configs/models/qwen3-0.6b.yml` setting embedding dimensions (`base_emb_dim: 1024`), intermediate MLP dimension (`base_mlp_dim: 3072`), attention query/KV heads (`16`/`8`), and number of layers (`28`). |
| **`steps`** | `50` | Defines training duration. | Runs the optimization/training loop for exactly 50 iterations, which is sufficient to compile the graph, calculate device step times, and measure TFLOPs throughput. |
| **`dataset_type`** | `synthetic` | Determines the data loader pipeline format. | Bypasses external dataset downloading and disk I/O bottlenecks by dynamically generating random tensors of input tokens on CPU memory. This allows pure computational throughput benchmarking. |
| **`base_output_directory`** | `./output` | Sets the file storage root path. | All metrics, TensorBoard execution logs, compile metadata, and model checkpoints are saved under this root directory. |
| **`run_name`** | `qwen3-0.6b-cpu` | Uniquely names the specific execution run. | Prevents overwriting other runs and creates nested folders named `qwen3-0.6b-cpu/` for output checkpoints and statistics. |

### 4. Execute CPU Training (Qwen 0.6B)

We run the main training script with our configuration overrides. We capture the stdout logs to parse metrics later.

In [14]:
!cat src/maxtext/configs/base.yml

# Copyright 2023–2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# This sentinel is a reminder to choose a real run name.
# If there is already a checkpoint under this run, that checkpoint will auto-resume.
#
# NOTE: Some unit/integration tests in MaxText do not always run this file directly.
# When running in decoupled mode (DECOUPLE_GCLOUD=TRUE), tests may use
# `decoupled_base_test.yml` instead of `base.yml` via `tests/utils/test_helpers.py`.
run_name: ""

model_name: "default"

In [15]:
# # Run training for 50 steps on CPU
# !python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
#     model_name=qwen3-0.6b \
#     steps=50 \
#     dataset_type=synthetic \
#     base_output_directory=./cpu-output \
#     run_name=qwen3-0.6b-cpu \
#     per_device_batch_size=1 \
#     attention=dot_product \
#     max_target_length=256 \
#     opt_type=sgd \
#     enable_checkpointing=false | tee train_cpu_run.log

In [4]:
!sudo apt-get update && sudo apt-get install -y \
    libxcomposite1 \
    libxcursor1 \
    libxdamage1 \
    libxext6 \
    libxfixes3 \
    libxi6 \
    libxrandr2 \
    libxrender1 \
    libxtst6 \
    libgbm1 \
    libasound2



Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:5 https://cli.github.com/packages stable InRelease                         
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information..

In [12]:
# Run training for 50 steps on CPU
!python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
    model_name=qwen3-0.6b \
    steps=50 \
    dataset_type=synthetic \
    base_output_directory=./cpu-output-bf16 \
    run_name=qwen3-0.6b-cpu-bf16 \
    per_device_batch_size=1 \
    attention=dot_product \
    max_target_length=256 \
    weight_dtype=bfloat16 \
    grad_dtype=bfloat16 \
    enable_checkpointing=false

2026-06-17 17:05:54.056013: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-17 17:05:54.105885: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-17 17:05:55.933256: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] mldiagnostics: dependency missing; using stub. (

In [11]:
!uv pip install aqtp ml-goodput-measurement ml-collections

Using Python 3.12.13 environment at: /usr
Resolved 56 packages in 112ms                                        
Prepared 1 package in 12ms                                               
Installed 1 package in 1ms.0                                
 + ml-collections==1.1.0


In [22]:
!uv pip install pathwaysutils aqt

Using Python 3.12.13 environment at: /usr
Resolved 68 packages in 273ms                                        
Prepared 10 packages in 4.52s                                            
Uninstalled 1 package in 3ms
Installed 10 packages in 149ms                              
 + anki==26.5
 + aqt==26.5
 - protobuf==5.29.6
 + protobuf==7.35.1
 + pyqt6==6.11.0
 + pyqt6-qt6==6.11.1
 + pyqt6-sip==13.11.1
 + pyqt6-webengine==6.11.0
 + pyqt6-webengine-qt6==6.11.1
 + truststore==0.10.4
 + waitress==3.0.2


In [18]:
!uv pip install -U transformers

Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 264ms                                        
Prepared 10 packages in 1.00s                                            
Uninstalled 10 packages in 773ms
Installed 10 packages in 121ms                              
 - anyio==4.13.0
 + anyio==4.14.0
 - certifi==2026.5.20
 + certifi==2026.6.17
 - filelock==3.29.2
 + filelock==3.29.4
 - fsspec==2025.3.0
 + fsspec==2026.6.0
 - huggingface-hub==1.18.0
 + huggingface-hub==1.19.0
 - numpy==2.0.2
 + numpy==2.4.6
 - regex==2025.11.3
 + regex==2026.5.9
 - rich==13.9.4
 + rich==15.0.0
 - tqdm==4.67.3
 + tqdm==4.68.3
 - transformers==5.10.2
 + transformers==5.12.1


In [17]:
!uv pip install drjax -q

In [2]:
!ls cpu-output-bf16/qwen3-0.6b-cpu/tensorboard/qwen3-0.6b-cpu/

ls: cannot access 'cpu-output-bf16/qwen3-0.6b-cpu/tensorboard/qwen3-0.6b-cpu/': No such file or directory


### 5. Parse Metrics & Logs

MaxText prints metric logs regularly during training. We will extract the compilation time, step times, loss, learning rate, and TFLOPs using a helper script.

In [40]:
%load_ext tensorboard


In [1]:
%tensorboard --logdir cpu-output/qwen3-0.6b-cpu/tensorboard/


UsageError: Line magic function `%tensorboard` not found.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls drive/

MyDrive  Othercomputers


In [13]:
!cp -r cpu-output-bf16 /content/drive/MyDrive/
